# DedrisGenAI on Google Colab

Run **DedrisGenAI** — an SDXL image generator with a clean **PHP web UI** — on a free Colab GPU, in one click.

This notebook starts the two local services that make up DedrisGenAI:

* the **Python + PyTorch engine** (`engine/server.py`) on `127.0.0.1:7866`, and
* the **PHP web UI** (`webui/`) on `127.0.0.1:8888`, which proxies to the engine.

It then exposes the UI through Colab's built-in port proxy (with a cloudflared fallback) so you can open it in your browser.

## How to run (one click)

1. **Set a GPU runtime:** **Runtime → Change runtime type → Hardware accelerator: GPU**.
2. **Run everything:** **Runtime → Run all** (or run the cells top to bottom).
3. When the last cells finish, **click the printed link** to open the UI.

> **First run downloads several GB** (PyTorch + the default SDXL model). The engine's first start can take a few minutes while the model downloads — this is normal and happens once per session.

> **Note on Enhance / auto-mask:** the *Enhance* tab and automatic mask generation rely on `rembg` (which pulls `pymatting` → `cupy`) and the SAM / GroundingDINO models. Colab's preinstalled `cupy` is often ABI-incompatible with NumPy, so those optional features may be unavailable here. This is **non-blocking** — plain text-to-image generation does not need them and works fine.

## 1. Clone the repository

In [ ]:
# Clone DedrisGenAI (idempotent: skip the clone if it is already present).
import os

REPO_URL = 'https://github.com/dedrisproject/DedrisGenAI.git'
REPO_DIR = '/content/DedrisGenAI'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned at', REPO_DIR)

%cd {REPO_DIR}
!git rev-parse --short HEAD

## 2. Install dependencies

Installs the **PHP CLI** (needed to serve the web UI), then **PyTorch** (CUDA build, matching Colab's GPU) and the engine's pinned Python requirements. All steps are idempotent — re-running is safe and fast once cached.

In [ ]:
# Install the PHP CLI — Colab does not ship PHP, and the web UI is served by `php -S`.
# apt is idempotent: a second run is a near-instant no-op.
!apt-get -qq update
!apt-get -qq install -y php-cli
!php --version

In [ ]:
# Install Python dependencies.
# Colab provides a CUDA GPU, so install the CUDA build of PyTorch from the official wheels.
!pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu121

# Then the rest of the engine requirements. These pin numpy==1.26.4 (so torch and
# the engine agree on the ABI). pip resolves this last, after torch is in place.
!pip install -q -r engine/requirements_versions.txt

# Quick sanity check that CUDA is visible to PyTorch.
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected — set Runtime → Change runtime type → GPU and re-run.')

## 3. Start the engine

Launches `engine/server.py` in the background (with the working directory set to `engine/`, as the engine requires), then waits for `127.0.0.1:7866/api/health` to report ready.

**The first start downloads the default model (several GB)** and primes caches, so health may take a few minutes to come up. Subsequent starts in the same session are fast. If the engine exits early, the cell prints the last log lines so you can see why.

In [ ]:
# Start the DedrisGenAI engine in the background. CWD must be `engine/`.
import os, sys, subprocess, time, json
from urllib.request import urlopen
from urllib.error import URLError

ENGINE_PORT = int(os.environ.get('DEDRIS_ENGINE_PORT', '7866'))
UI_PORT = int(os.environ.get('DEDRIS_UI_PORT', '8888'))
os.environ['DEDRIS_ENGINE_PORT'] = str(ENGINE_PORT)
os.environ['DEDRIS_UI_PORT'] = str(UI_PORT)

ENGINE_LOG = '/content/dedris_engine.log'


def engine_healthy():
    """Return the /api/health JSON if the engine is up, else None."""
    try:
        with urlopen(f'http://127.0.0.1:{ENGINE_PORT}/api/health', timeout=3) as r:
            return json.load(r)
    except (URLError, OSError, ValueError):
        return None


# Idempotent: only start a new engine if one is not already healthy.
if engine_healthy():
    print('Engine already running and healthy.')
else:
    print('Starting engine (first start downloads the default model — this can take a few minutes)...')
    with open(ENGINE_LOG, 'wb') as log:
        engine_proc = subprocess.Popen(
            [sys.executable, 'server.py'],
            cwd='engine',
            stdout=log,
            stderr=subprocess.STDOUT,
        )

    # Wait for /api/health. Generous timeout to cover the one-time model download.
    health = None
    deadline = time.time() + 1800  # 30 minutes
    while time.time() < deadline:
        health = engine_healthy()
        if health:
            break
        if engine_proc.poll() is not None:
            print('Engine process exited early. Last log lines:')
            !tail -n 40 {ENGINE_LOG}
            raise SystemExit('Engine failed to start — see log above.')
        print('  ...waiting for engine (downloading model on first run)', flush=True)
        time.sleep(10)

    if not health:
        !tail -n 40 {ENGINE_LOG}
        raise SystemExit('Engine did not become healthy in time — see log above.')
    print('Engine ready:', health)

## 4. Start the PHP web UI

Serves the PHP UI on `127.0.0.1:8888`. The browser only ever talks to this PHP server; it proxies all `/api/*` calls to the engine.

In [ ]:
# Start the PHP web UI in the background on the UI port.
import subprocess, time, os
from urllib.request import urlopen
from urllib.error import URLError

UI_LOG = '/content/dedris_ui.log'


def ui_up():
    """True if the PHP server is listening (any HTTP response counts)."""
    try:
        urlopen(f'http://127.0.0.1:{UI_PORT}/', timeout=3)
        return True
    except URLError as e:
        # An HTTP error response still means the server is listening.
        return getattr(e, 'code', None) is not None
    except OSError:
        return False


if ui_up():
    print('PHP UI already running.')
else:
    print(f'Starting PHP UI on 127.0.0.1:{UI_PORT}...')
    with open(UI_LOG, 'wb') as log:
        ui_proc = subprocess.Popen(
            ['php', '-S', f'127.0.0.1:{UI_PORT}', '-t', 'webui/public', 'webui/public/router.php'],
            stdout=log,
            stderr=subprocess.STDOUT,
        )
    for _ in range(30):
        if ui_up():
            break
        time.sleep(1)
    if not ui_up():
        !tail -n 40 {UI_LOG}
        raise SystemExit('PHP UI did not start — see log above.')
    print('PHP UI is up.')

## 5. Open the UI

Colab cannot expose `localhost` directly, so we publish the UI port through Colab's built-in **port proxy**. Run the cell below and click the printed URL.

If the proxy link does not work for you (some browsers / corporate networks block it), use the **cloudflared tunnel fallback** in the next cell instead.

In [ ]:
# Expose the UI via Colab's port proxy and print a clickable link.
from google.colab.output import eval_js

proxy_url = eval_js(f'google.colab.kernel.proxyPort({UI_PORT})')
print('Open DedrisGenAI here:')
print(proxy_url)

from IPython.display import HTML, display
display(HTML(f'<a href="{proxy_url}" target="_blank" rel="noopener">▶ Open DedrisGenAI UI</a>'))

### Fallback: cloudflared tunnel (optional)

Only run this if the Colab port-proxy link above did not work. It downloads `cloudflared` and opens a temporary public tunnel to the UI port. Look for the printed `https://<random>.trycloudflare.com` URL (treat it as public for the life of the session).

In [ ]:
# Cloudflared tunnel fallback. Run ONLY if the Colab proxy link did not work.
import os, subprocess

CF_BIN = '/content/cloudflared'
if not os.path.exists(CF_BIN):
    !wget -q -O {CF_BIN} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x {CF_BIN}

CF_LOG = '/content/cloudflared.log'
with open(CF_LOG, 'wb') as log:
    subprocess.Popen([CF_BIN, 'tunnel', '--url', f'http://127.0.0.1:{UI_PORT}'],
                     stdout=log, stderr=subprocess.STDOUT)

# Give cloudflared a moment to register, then surface the public URL.
import time, re
url = None
for _ in range(30):
    time.sleep(2)
    try:
        with open(CF_LOG) as f:
            m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', f.read())
        if m:
            url = m.group(0)
            break
    except FileNotFoundError:
        pass

if url:
    print('Public tunnel URL:', url)
    from IPython.display import HTML, display
    display(HTML(f'<a href="{url}" target="_blank" rel="noopener">▶ Open DedrisGenAI UI (tunnel)</a>'))
else:
    print('Could not find the tunnel URL yet. Check the log:')
    !tail -n 30 {CF_LOG}

---

**Tips**

* Engine logs: `!tail -n 50 /content/dedris_engine.log`
* UI logs: `!tail -n 50 /content/dedris_ui.log`
* The **first generation** downloads the default SDXL model (GBs) on demand if it wasn't fetched at startup — give it a moment.
* The **Enhance / auto-mask** features need `rembg` + `cupy`, which may be unavailable on Colab; plain text-to-image is unaffected.
* Keep this tab open — Colab stops the runtime (and both services) when the session ends.
* See `docs/COLAB.md` in the repo for troubleshooting (GPU runtime, tunnel fallback, model download time).